# 🎥 ROTBOTS — Video Generator
## From Prompts to Video Clips

**Workshop: "Anatomy of a Content Machine"**  
LI-MA Transformation Digital Art 2026, Amsterdam

---

This notebook takes the `prompts.json` from the Script Writer notebook and:

1. **Generate** — Send each prompt to fal.ai to create video clips
2. **Found Footage** — Optionally upload your own video clips instead
3. **Preview** — Watch and compare all clips before assembly

> **🤔 Think about:** What does it cost (financially, environmentally) to generate these clips?

---

## 🔧 Setup

Connects to Google Drive to load `prompts.json` from the Script Writer notebook.

In [ ]:
# ============================================================
# SETUP — Run this cell first!
# ============================================================

!pip install -q fal-client requests

import fal_client
import json
import os
import time
import requests
from pathlib import Path
from IPython.display import display, Markdown, HTML, Video

# Connect to Google Drive (shared workspace between notebooks)
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
WORK_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = WORK_DIR / 'videos'
VIDEOS_DIR.mkdir(exist_ok=True)

print(f'✅ Connected to Google Drive')
print(f'📁 Shared workspace: {WORK_DIR}')
print()
print('Files found from previous notebooks:')
for f in sorted(WORK_DIR.glob('*.json')):
    print(f'   ✅ {f.name} ({f.stat().st_size / 1024:.1f}KB)')

In [ ]:
# ============================================================
# API KEY — Paste your fal.ai API key here
# Get one at: https://fal.ai/dashboard/keys
# ============================================================

FAL_API_KEY = ""  # <-- Paste your key between the quotes

if not FAL_API_KEY:
    print("⚠️  Please paste your fal.ai API key above!")
    print("   Get one at: https://fal.ai/dashboard/keys")
else:
    os.environ["FAL_KEY"] = FAL_API_KEY
    print("✅ fal.ai API key configured")

In [ ]:
# ============================================================
# LOAD PROMPTS from Google Drive
# ============================================================

prompts_file = WORK_DIR / 'prompts.json'

if prompts_file.exists():
    with open(prompts_file) as f:
        prompts = json.load(f)
    print(f'✅ Loaded {len(prompts)} prompts from Google Drive')
    for p in prompts:
        print(f"   Scene {p['scene']}: {p['title']} [{p['style']}] ({p['duration']}s)")
else:
    print('❌ No prompts.json found on Google Drive!')
    print('   Run the Script Writer notebook (02_Script_Writer.ipynb) first.')
    print(f'   Expected location: {prompts_file}')

---
## 🎬 Station 5: Generate Videos

Each clip takes **1-4 minutes** to generate.

> **🤔 Discussion:**
> - Each 5-second clip costs roughly $0.25-$0.50. A 30-second video = ~$2-3.
> - The GPU consumes significant electricity. What's the carbon footprint?
> - Who owns this generated footage?

In [ ]:
# ============================================================
# VIDEO GENERATION SETTINGS
# ============================================================

VIDEO_MODELS = {
    "wan-2.2": {"endpoint": "fal-ai/wan/v2.2/1.3b/text-to-video", "description": "Wan 2.2 1.3B — Fast, cheapest (~$0.05/s)", "default_seconds": 5},
    "minimax-standard": {"endpoint": "fal-ai/minimax/video-01", "description": "MiniMax Hailuo 01 — Higher quality, ~$0.28/video", "default_seconds": 6},
    "ltx-2.3": {"endpoint": "fal-ai/ltx-video/v2.3", "description": "LTX 2.3 — Fast, good prompt adherence", "default_seconds": 5},
}

CHOSEN_MODEL = "wan-2.2"  # Options: wan-2.2, minimax-standard, ltx-2.3

model = VIDEO_MODELS[CHOSEN_MODEL]
print(f"🎥 Model: {CHOSEN_MODEL} — {model['description']}")
print(f"   Estimated cost for {len(prompts)} scenes: ~${len(prompts) * 0.25:.2f} - ${len(prompts) * 0.50:.2f}")

In [ ]:
# ============================================================
# GENERATE VIDEOS — Takes a few minutes per scene!
# ============================================================

print(f"🎬 Generating {len(prompts)} clips with {CHOSEN_MODEL}...")
print(f"   Estimated time: {len(prompts) * 2}-{len(prompts) * 4} minutes")
print("=" * 60)

generated_videos = []

for i, prompt_data in enumerate(prompts):
    scene_num = prompt_data['scene']
    t2v_prompt = prompt_data['t2v_prompt']
    duration = prompt_data.get('duration', model['default_seconds'])

    print(f"\n🎥 Scene {scene_num} ({i+1}/{len(prompts)}): {prompt_data['title']}")
    print(f"   Prompt: {t2v_prompt[:100]}...")
    print(f"   Generating ({duration}s)...", end=' ', flush=True)

    start_time = time.time()
    try:
        input_data = {'prompt': t2v_prompt}
        if CHOSEN_MODEL == 'wan-2.2':
            input_data['num_frames'] = duration * 16 + 1
            input_data['aspect_ratio'] = '16:9'
            input_data['enable_prompt_expansion'] = False
        elif CHOSEN_MODEL == 'ltx-2.3':
            input_data['num_frames'] = duration * 24 + 1
            input_data['aspect_ratio'] = '16:9'

        result = fal_client.subscribe(model['endpoint'], arguments=input_data)
        elapsed = time.time() - start_time

        video_url = result.get('video', {}).get('url', '') or result.get('url', '')
        if video_url:
            video_path = VIDEOS_DIR / f'scene_{scene_num:03d}.mp4'
            resp = requests.get(video_url, timeout=120)
            with open(video_path, 'wb') as f: f.write(resp.content)
            size_kb = video_path.stat().st_size / 1024
            generated_videos.append({'scene': scene_num, 'title': prompt_data['title'], 'path': str(video_path), 'duration': duration, 'source': 'generated'})
            print(f'✅ Done! ({elapsed:.0f}s, {size_kb:.0f}KB)')
        else:
            print(f'⚠️ No video URL in response')
    except Exception as e:
        print(f'❌ Error: {e}')

print(f"\n{'='*60}")
print(f'✅ Generated {len(generated_videos)}/{len(prompts)} videos')
print(f'📁 Saved to Google Drive: {VIDEOS_DIR}')

---
## 📂 Found Footage (Optional)

Upload your own MP4 clips to mix with or replace AI-generated ones.

In [ ]:
# UPLOAD FOUND FOOTAGE (Optional)
USE_FOUND_FOOTAGE = False  # Set to True to upload

if USE_FOUND_FOOTAGE:
    from google.colab import files
    print('📂 Upload MP4 files (name them scene_002.mp4 etc. to replace specific scenes):')
    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = VIDEOS_DIR / filename
        with open(dest, 'wb') as f: f.write(content)
        scene_num = int(''.join(filter(str.isdigit, filename.split('.')[0]))) if any(c.isdigit() for c in filename) else 100 + len(generated_videos)
        generated_videos.append({'scene': scene_num, 'title': f'Found: {filename}', 'path': str(dest), 'duration': 0, 'source': 'found_footage'})
        print(f'   ✅ {filename} ({len(content)/1024:.0f}KB)')
else:
    print('ℹ️ Set USE_FOUND_FOOTAGE = True above to upload your own clips.')

---
## 👀 Preview Videos

In [ ]:
# PREVIEW ALL VIDEOS
generated_videos.sort(key=lambda x: x['scene'])
print(f'🎬 {len(generated_videos)} video clips:')
for v in generated_videos:
    tag = '🤖 AI' if v['source'] == 'generated' else '📂 Found'
    display(Markdown(f"### Scene {v['scene']}: {v['title']} — {tag}"))
    if Path(v['path']).exists():
        try: display(Video(v['path'], embed=True, width=640))
        except: print(f'   (File at: {v["path"]})')
    display(Markdown('---'))

In [ ]:
# SAVE MANIFEST
manifest = {'model': CHOSEN_MODEL, 'videos': generated_videos}
with open(WORK_DIR / 'video_manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
print(f'✅ Manifest saved to Google Drive ({len(generated_videos)} clips)')
print(f'\n📁 Videos on Drive:')
for f in sorted(VIDEOS_DIR.glob('*.mp4')): print(f'   {f.name} ({f.stat().st_size/1024:.0f}KB)')
print('\n→ Next: 04_The_Voice.ipynb for narration, then 06_Assemble.ipynb to combine everything!')

---
## ⏭️ Next Steps

Videos are saved on Google Drive. Next:
- **04_The_Voice.ipynb** — Generate narration audio
- **06_Assemble.ipynb** — Combine videos + audio into final video

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*